# Laptop lab 2: Nearest neighbors and linear regression
*Predict a price, and test it honestly*

**Machine Learning with Python** · Minor Amsterdam Data Science and Artificial Intelligence · Fall 2026<br>
Thursday, November 5, 2026 · Laptop labs: Batuhan Ataş (b.atas@uva.nl)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/batuatas/MLwPython-labs/blob/main/lab02.ipynb)

> **Before you start:** choose **File → Save a copy in Drive** and work in that copy. Colab does not save your changes in the original notebook.

In Lab 1 your friend asked what she can ask per night for her apartment in Amsterdam, and you cleaned the Airbnb data to find out. You ended with a first answer: the median price of *similar* listings. That was already almost a model. This week, Lecture 3 gave you your first real model, $k$-nearest neighbors, and a recipe to test any model honestly; Lecture 4 gave you linear regression. Today you use both on the same data.

1. **Whole home or private room?** (Task 1): a first classifier with $k$-NN, and the effect of $k$.
2. **Choosing $k$ honestly** (Task 2): cross-validation and grid search.
3. **A price from the neighbors** (Task 3): $k$-NN for a number.
4. **A line through the data** (Task 4): linear regression, $R^2$, residuals and outliers.
5. **Many features, few listings** (Task 5): ridge and lasso.
6. **Back to your friend** (Task 6): her price, and how sure we are.

## How this lab works

As in Lab 1, most of the code is given: read it, run it, and look carefully at the output. In some cells you fill in the dots `...` yourself, often with a line from the slides.

| You see | What you do |
|---|---|
| **Run** | Run the cell and look at the output. |
| **Your turn** | Fill in the dots `...`. Until you do, the cell shows an orange *Not yet* box: that is normal. |
| **Check** | Run it right after your own cell. It says ✓ when your code works, or tells you what to look at. |
| **Hint** | Click to unfold it when you are stuck. |
| **Think** | Double-click *Your answer* and write a sentence or two. We discuss the answers together. |

Run the cells from top to bottom with **Shift + Enter**: later cells use the results of earlier ones. Some cells take a few seconds: a grid search fits hundreds of models. Stuck for more than five minutes? Raise your hand.

Every model in scikit-learn works the same way (Lecture 3):

```python
model = SomeModel(setting=...)   # choose the model and its hyperparameters
model.fit(X_train, y_train)      # learn from the training set
model.predict(X_test)            # predictions for new examples
model.score(X_test, y_test)      # accuracy (classification) or R² (regression)
```

| Task | Minutes |
|---|---|
| 0. Setup | 5 |
| 1. Whole home or private room? | 15 |
| 2. Choosing $k$ honestly | 15 |
| 3. A price from the neighbors | 10 |
| 4. A line through the data | 20 |
| 5. Many features, few listings | 15 |
| 6. Back to your friend | 5 |
| Check your understanding | together, at the end |
| If you finish early | at home |

## Task 0. Setup

Run the cell below. It loads the libraries, downloads the checks of this lab and the data from Inside Airbnb, and repeats the cleaning of Lab 1, so that everybody starts from the same table. You should see a green box with a ✓.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LinearRegression, HuberRegressor, Ridge, Lasso

# The checks of this lab are in the course lab repository;
# setup_ok() also downloads the data from Inside Airbnb.
BASE = "https://raw.githubusercontent.com/batuatas/MLwPython-labs/main/"
import urllib.request
urllib.request.urlretrieve(BASE + "lab02_tools.py", "lab02_tools.py")
from lab02_tools import *

setup_ok()

**Run.** The clean table of Lab 1. As a reminder, this is what the cleaning did:

| Problem | What we did |
|---|---|
| prices stored as text | removed `$` and `,`, converted to numbers |
| 43% of the prices missing | dropped those listings |
| `maximum_nights` of 2,147,483,647 | replaced by `np.nan` |
| a few prices of €2,000 or more | left out: not comparable with an ordinary apartment |

In [ ]:
clean = load_clean()
print(len(clean), "listings")
clean.head()

## Task 1. Whole home or private room?

Lecture 3 started with a classification problem: passed or failed, from study hours and social media. We start with one on Airbnb: can a model tell a **whole home** from a **private room**, knowing only its price and its number of reviews? (Hotel rooms and shared rooms are rare, so we leave them out.)

**Run.** The two features, with the label as the color: the plot "The task" of Lecture 3, for Airbnb.

In [ ]:
two = clean[clean["room_type"].isin(["Entire home/apt", "Private room"])]   # keep two room types

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.scatterplot(data=two, x="number_of_reviews", y="price", hue="room_type",
                alpha=0.4, s=12, linewidth=0, ax=ax)   # hue: one color per room type
ax.set_xlim(0, 800)       # a few listings have thousands of reviews; we zoom in
ax.set_xlabel("number of reviews")
ax.set_ylabel("price per night (euros)")
plt.show()

**Think.** Where are the private rooms, and where the whole homes? Can you think of a reason why private rooms collect so many more reviews?

*Your answer:*

### Training and test data

**Run.** Features `X`, label `y`, and the split of Lecture 3: 75% of the listings to learn from, 25% put away to test on. `stratify=y` keeps the same share of private rooms in both parts, and `random_state=0` makes the split the same for everybody.

In [ ]:
X = two[["price", "number_of_reviews"]]     # the features: a table, so double brackets
y = two["room_type"]                         # the label: one column

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=0)
print(len(X_train), "listings to train on,", len(X_test), "to test on")
y.value_counts(normalize=True)               # the share of each room type

**Your turn.** Train $k$-NN with $k = 5$ on the training set, and compute its accuracy on the test set. Follow the slide "k-NN in scikit-learn" of Lecture 3: fit, predict, score.

In [ ]:
knn = KNeighborsClassifier(n_neighbors=...)
knn.fit(..., ...)                       # learn from the training set
print(knn.predict(X_test)[:5])          # the predicted labels of the first five test listings
knn.score(..., ...)                     # accuracy on the test set

<details><summary><b>Hint</b> (click to unfold)</summary>

The slide of Lecture 3:

```python
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
knn.score(X_test, y_test)
```

</details>

In [ ]:
# Check
check_1a()

**Think.** 72% of the listings are whole homes. What accuracy does a "model" get that always says *whole home*, without looking at the features? What does that tell you about the accuracy of $k$-NN? (Remember the fraud question at the end of Lab 1.)

*Your answer:*

### The effect of $k$

**Run.** For every $k$ from 1 to 100, the accuracy on the training set and on the test set: the plot "Accuracy against $k$" of Lecture 3. It fits 100 models, so it takes a few seconds.

In [ ]:
ks = range(1, 101)
train_acc, test_acc = [], []
for k in ks:
    model = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    train_acc.append(model.score(X_train, y_train))
    test_acc.append(model.score(X_test, y_test))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ks, train_acc, label="training data")
ax.plot(ks, test_acc, label="test data")
ax.axhline(0.72, color="grey", linestyle="--", label="always 'whole home'")   # the baseline
ax.set_xlabel("number of neighbors k")
ax.set_ylabel("accuracy")
ax.legend()
print("k = 1: training", round(train_acc[0], 3), " test", round(test_acc[0], 3))
plt.show()

**Think.** (a) Why is the training accuracy of 1-NN almost 100%, and why is it not exactly 100%? (b) Where do you see overfitting, and where underfitting (Lecture 3, "The sweet spot")? (c) Which $k$ would you choose from this plot? Hold on to your answer: Task 2 explains why we should not choose it this way.

*Your answer:*

### Distance depends on the units

$k$-NN finds the nearest neighbors by distance. Lecture 3 showed that the units decide what "near" means: in minutes, study time dominated the distance. Here, prices run up to 2,000 euros and numbers of reviews up to several thousands.

**Your turn.** Make a copy of `X` with the price in **thousands of euros** instead of euros (divide by 1000), and train the same $k$-NN with $k = 5$ on it. The rest is given.

In [ ]:
X_k = X.assign(price=X["price"] / ...)        # the price in thousands of euros
X_train_k, X_test_k = X_k.loc[X_train.index], X_k.loc[X_test.index]   # the same split as before
knn_k = KNeighborsClassifier(n_neighbors=5).fit(X_train_k, y_train)
accuracy_thousands = knn_k.score(X_test_k, y_test)
accuracy_thousands

In [ ]:
# Check
check_1c()

**Think.** The data are the same, only the unit of the price changed. Why does the accuracy change? Which feature does the model mostly look at now?

*Your answer:*

## Task 2. Choosing $k$ honestly

In Task 1 you could pick the $k$ with the best test accuracy: $k = 34$, 85.6%. Lecture 3, "Do not choose $k$ on the test set": once you choose anything with the test set, it has become training data, and 85.6% is too optimistic for new listings. Instead, we choose $k$ with **cross-validation** on the training set, and keep the test set for one final check.

**Your turn.** Compute the five cross-validation accuracies of $k$-NN with $k = 5$ on the training set, as on the slide "Cross-validation in scikit-learn", and their mean.

In [ ]:
knn5 = KNeighborsClassifier(n_neighbors=5)
scores = cross_val_score(knn5, ..., ..., cv=...)   # on the training set only
print(scores)            # five accuracies, one per fold
scores.mean()            # their average

<details><summary><b>Hint</b> (click to unfold)</summary>

`cross_val_score(model, X_train, y_train, cv=5)`: the model, the training features, the training labels, and the number of folds.

</details>

In [ ]:
# Check
check_2a()

**Think.** The five accuracies are not equal. What does that tell you about a single validation set? And why are the folds of a classifier *stratified* (Lecture 3, "Defaults worth knowing")?

*Your answer:*

**Your turn.** Now try every $k$ from 1 to 60 with a grid search, as on the slide "Grid search: try every k". Fill in the range of $k$, the number of folds, and the data to fit on.

In [ ]:
param_grid = {"n_neighbors": range(1, ...)}           # every k from 1 to 60
grid = GridSearchCV(KNeighborsClassifier(), param_grid, cv=...)
grid.fit(..., ...)                                    # the training set only

print(grid.best_params_)        # the k with the best mean cross-validation accuracy
print(grid.best_score_)         # that mean accuracy

<details><summary><b>Hint</b> (click to unfold)</summary>

`range(1, 61)` stops *before* 61, so it runs from 1 to 60. Use `cv=5`, and fit on `X_train, y_train`.

</details>

In [ ]:
# Check
check_2b()

**Run.** The mean cross-validation accuracy of every $k$, with a band of ± one standard deviation over the five folds.

In [ ]:
res = pd.DataFrame(grid.cv_results_)
k_values = res["param_n_neighbors"].astype(int)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_values, res["mean_test_score"], label="mean over 5 folds")
ax.fill_between(k_values, res["mean_test_score"] - res["std_test_score"],
                res["mean_test_score"] + res["std_test_score"], alpha=0.2, label="± one std. dev.")
ax.set_xlabel("number of neighbors k")
ax.set_ylabel("cross-validation accuracy")
ax.legend()
plt.show()

**Run.** Step 3 and 4 of "The full recipe": `GridSearchCV` has already refit the best $k$ on the whole training set. Now, and only now, we use the test set: once.

In [ ]:
grid.score(X_test, y_test)      # the number we report

**Think.** Compare three numbers: the best test accuracy of Task 1 (85.6% at $k = 34$), the cross-validation accuracy of the grid search, and its test accuracy. Which one would you report as the accuracy on new listings, and why not the 85.6%?

*Your answer:*

> **The full recipe** (Lecture 3): 1. Split off a test set, and put it away. 2. Choose hyperparameters with cross-validation on the training set (`GridSearchCV`). 3. Refit the best model on the whole training set. 4. Score it once on the test set, and report that number. We use this recipe for every model in the rest of the course, starting now.

## Task 3. A price from the neighbors

Back to your friend's question: the price is a number, so this is regression. Lecture 3, "k-NN for regression": predict the *average* price of the $k$ nearest listings, with `KNeighborsRegressor`.

For a price, the location matters. Latitude and longitude are in degrees, and around Amsterdam a degree of longitude is about 68 km, a degree of latitude about 111 km: distances in degrees would distort the map (that is why the maps of Lab 1 needed `set_aspect`). So we turn the location into kilometers east and north of Dam Square, the center of the city.

**Run.** The features of the model: the location in km, the number of guests, and whether the listing is a whole home (1) or not (0).

In [ ]:
clean["x_km"] = (clean["longitude"] - 4.8913) * 68.0     # km east of Dam Square
clean["y_km"] = (clean["latitude"] - 52.3731) * 111.2    # km north of Dam Square
clean["whole_home"] = (clean["room_type"] == "Entire home/apt").astype(int)   # 1 or 0

X = clean[["x_km", "y_km", "accommodates", "whole_home"]]
y = clean["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)
print(len(X_train), "listings to train on,", len(X_test), "to test on")

From here on, `X` and `y` are about the price. (If you go back to Task 1 or 2, run their cells from the top again.)

**Your turn.** Follow the full recipe of Task 2, now with `KNeighborsRegressor`: a grid search over every $k$ from 1 to 60, with 5-fold cross-validation, on the training set.

For a regressor, `score` gives $R^2$ instead of accuracy (Lecture 4, "R²: how much better than the mean?"): 1 is perfect, 0 is no better than always predicting the mean price.

In [ ]:
grid_knn = GridSearchCV(..., {"n_neighbors": range(1, 61)}, cv=5)
grid_knn.fit(..., ...)
print(grid_knn.best_params_, grid_knn.best_score_)    # best k, and its mean cross-validation R²
grid_knn.score(X_test, y_test)                        # R² on the test set

<details><summary><b>Hint</b> (click to unfold)</summary>

The same as in Task 2, with `KNeighborsRegressor()` instead of `KNeighborsClassifier()`, fitted on `X_train, y_train`.

</details>

In [ ]:
# Check
check_3()

**Run.** Your friend's apartment: near the Sarphatipark in De Pijp, a whole home for 2 guests. The model finds her nearest neighbors in the training set and averages their prices.

In [ ]:
her_lat, her_lon = 52.3540, 4.8960
her = pd.DataFrame({"x_km": [(her_lon - 4.8913) * 68.0],
                    "y_km": [(her_lat - 52.3731) * 111.2],
                    "accommodates": [2], "whole_home": [1]})
grid_knn.predict(her)

**Think.** In Lab 1 you took the median of similar listings in her neighborhood: €300. $k$-NN predicts a bit more. What does $k$-NN do differently from your filter in Lab 1? And the test $R^2$ is about 0.41: is that good?

*Your answer:*

## Task 4. A line through the data

Lecture 4: a linear model gives every feature one coefficient, and predicts

$$\hat{y} = w_1 x_1 + w_2 x_2 + \dots + w_p x_p + b.$$

Least squares chooses the coefficients $w$ and the intercept $b$ with the smallest sum of squared gaps between the actual and the predicted prices.

**Run.** Seven features, including the distance to Dam Square. Linear regression in scikit-learn cannot handle missing values, and 856 listings have a gap in one of these columns (mostly listings without reviews, so without a rating). We drop them; Lecture 7 shows better ways.

In [ ]:
clean["dist_center_km"] = np.sqrt(clean["x_km"]**2 + clean["y_km"]**2)   # km to Dam Square

features = ["accommodates", "bedrooms", "beds", "whole_home",
            "review_scores_rating", "number_of_reviews", "dist_center_km"]
model_data = clean.dropna(subset=features)      # only the listings without gaps in these columns
print(len(model_data), "listings")

X = model_data[features]
y = model_data["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

### One feature

**Your turn.** Fit a linear regression of the price on the number of guests only, as on the slide "Linear regression in scikit-learn". Use double brackets, `X_train[["accommodates"]]`: scikit-learn expects a table of features, even with one column.

In [ ]:
lr = LinearRegression().fit(..., ...)
print("slope:", lr.coef_, " intercept:", lr.intercept_)

In [ ]:
# Check
check_4a()

**Think.** Read the model in words, as on the slide "A straight line through the data". In Lab 1, the line after cleaning gave about 76 euros per extra guest: close? What does the intercept mean here, and does it make sense?

*Your answer:*

### $R^2$: how much better than the mean?

**Run.** The baseline of Lecture 4 always predicts the mean price. $R^2$ compares the squared errors of the model with those of this baseline, on the same listings: here the test listings.

In [ ]:
baseline = np.full(len(y_test), y_test.mean())           # always the mean price
predicted = lr.predict(X_test[["accommodates"]])

ss_model = ((y_test - predicted)**2).sum()
ss_mean = ((y_test - baseline)**2).sum()
print("R² by hand:  ", 1 - ss_model / ss_mean)
print("R² by score: ", lr.score(X_test[["accommodates"]], y_test))

**Think.** What does an $R^2$ of about 0.24 mean in words? Can the $R^2$ on the test set be negative?

*Your answer:*

### Several features

**Your turn.** Now fit a linear regression on all seven features, and call it `lr7`.

In [ ]:
lr7 = ...
lr7.score(X_test, y_test)       # R² on the test set

In [ ]:
# Check
check_4c()

**Run.** The coefficients, one per feature.

In [ ]:
pd.Series(lr7.coef_, index=features).round(1)

**Think.** Read three coefficients with care (Lecture 4, "Reading coefficients with care"). (a) What does the coefficient of `dist_center_km` say? (b) The coefficient of `beds` is negative: does an extra bed make a listing cheaper? (c) Does the coefficient of `review_scores_rating` prove that a better rating *causes* a higher price?

*Your answer:*

### Check the residuals

A residual is the actual price minus the predicted price. Lecture 4: a good model leaves "a shapeless cloud around 0"; a curve, a funnel, or a few far-away points are a warning.

**Your turn.** Plot the predicted prices of the test listings on the x-axis against their residuals on the y-axis, with transparent dots, a horizontal line at 0, and labels on both axes.

In [ ]:
predicted = lr7.predict(X_test)
residuals = y_test - predicted          # actual minus predicted

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(..., ..., alpha=0.3, s=10)
ax.axhline(0, color="black")            # the line at 0
ax.set_xlabel(...)
ax.set_ylabel(...)
plt.show()

In [ ]:
# Check
check_4d()

**Think.** Which warning sign of the slide do you see? What does it mean for your friend? Also look at the far left: what is strange about some predictions?

*Your answer:*

### Outliers pull the line

In Lab 1 we left out 10 listings of €2,000 a night or more. Lecture 4, "Outliers pull the line": squared gaps make far-away points very influential. The Huber loss counts large gaps less (`HuberRegressor`).

**Run.** The line of the price against the number of guests, fitted three ways.

In [ ]:
priced = load_priced()                 # with the 10 outliers of Lab 1
fits = {
    "least squares, with the outliers": LinearRegression().fit(priced[["accommodates"]], priced["price"]),
    "Huber, with the outliers": HuberRegressor(max_iter=1000).fit(priced[["accommodates"]], priced["price"]),
    "least squares, clean data": LinearRegression().fit(clean[["accommodates"]], clean["price"]),
}
for name, model in fits.items():
    print(f"{name:35s} {model.coef_[0]:6.1f} euros per extra guest")

**Think.** Ten listings of 6,039 change the slope by about 16 euros per guest. What does Huber do with them? Which line would you trust, and which question should you ask first?

*Your answer:*

## Task 5. Many features, few listings

Lecture 4 added 80 survey questions that had nothing to do with the grade, and the model began to fit noise. Here we do something similar with real features: the neighborhood (22 values) and the property type (57 values, such as *Entire rental unit* or *Boat*).

A model needs numbers, so we turn each of these text columns into many 0/1 columns, one per value: `neighbourhood_Zuid` is 1 for listings in Zuid and 0 for all others. `pd.get_dummies` does that for you (Lecture 6 explains it properly). Together with the seven features of Task 4, that gives 86 features.

And we pretend that we only have **200 listings** to learn from: `train_size=200`. All the other listings are the test set.

**Run.**

In [ ]:
dummies = pd.get_dummies(model_data[["neighbourhood", "property_type"]], dtype=float)
X_wide = pd.concat([model_data[features], dummies], axis=1)
print(X_wide.shape[1], "features")

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(X_wide, y, train_size=200, random_state=0)
lr_wide = LinearRegression().fit(X_train_w, y_train_w)
print("training R²:", round(lr_wide.score(X_train_w, y_train_w), 2))
print("test R²:    ", round(lr_wide.score(X_test_w, y_test_w), 2))

**Think.** Compare the training and the test $R^2$. What happened? Why does it happen with 86 features and 200 listings?

*Your answer:*

### Ridge: keep the coefficients small

Ridge adds a penalty $\alpha \sum_j w_j^2$ to the squared gaps: a larger $\alpha$ gives smaller coefficients and a simpler model. $\alpha$ is a hyperparameter, so we choose it with the recipe of Task 2.

**Your turn.** A grid search over $\alpha$, as on the slide "Choosing α": 25 values from 0.01 to 10,000, evenly spaced on a log scale.

In [ ]:
param_grid = {"alpha": np.logspace(..., ..., 25)}      # 25 values from 10^-2 to 10^4
grid_ridge = GridSearchCV(Ridge(), param_grid, cv=5)
grid_ridge.fit(X_train_w, y_train_w)
print(grid_ridge.best_params_)
grid_ridge.score(X_test_w, y_test_w)                   # R² on the test set

<details><summary><b>Hint</b> (click to unfold)</summary>

`np.logspace(-2, 4, 25)`: from $10^{-2} = 0.01$ to $10^{4} = 10{,}000$, 25 values.

</details>

In [ ]:
# Check
check_5a()

### Lasso: remove features

Lasso uses the penalty $\alpha \sum_j |w_j|$ instead. Lecture 4: "Most coefficients become exactly zero: lasso selects features."

**Your turn.** The same grid search with `Lasso`, with 17 values of $\alpha$ from 0.01 to 100. Then the cell lists the features that lasso keeps.

In [ ]:
param_grid = {"alpha": np.logspace(-2, 2, 17)}         # 17 values from 0.01 to 100
grid_lasso = GridSearchCV(..., param_grid, cv=5)
grid_lasso.fit(X_train_w, y_train_w)
print(grid_lasso.best_params_, " test R²:", round(grid_lasso.score(X_test_w, y_test_w), 2))

coef = pd.Series(grid_lasso.best_estimator_.coef_, index=X_wide.columns)
coef[coef != 0].sort_values().round(1)                 # the features lasso keeps

<details><summary><b>Hint</b> (click to unfold)</summary>

`Lasso()` in place of the dots. If you see a *ConvergenceWarning*, use `Lasso(max_iter=100000)`: lasso then takes more steps to find its coefficients.

</details>

In [ ]:
# Check
check_5b()

**Think.** How many of the 86 features does lasso keep, and which neighborhoods are among them? Does the list make sense, given your maps of Lab 1? Lecture 4 asked: "Many features, few of them useful? Try lasso. Many small effects? Try ridge." Which describes these data, judging by the test scores?

*Your answer:*

### More data also helps

**Run.** The same 86 features, now with more and more training listings. For each size, the test $R^2$ of plain linear regression and of ridge with $\alpha = 10$.

In [ ]:
sizes = [200, 500, 1000, 2000, 3500]
r2_lr, r2_ridge = [], []
for n in sizes:
    Xa, Xb, ya, yb = train_test_split(X_wide, y, train_size=n, random_state=0)
    r2_lr.append(LinearRegression().fit(Xa, ya).score(Xb, yb))
    r2_ridge.append(Ridge(alpha=10).fit(Xa, ya).score(Xb, yb))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sizes, r2_lr, marker="o", label="linear regression")
ax.plot(sizes, r2_ridge, marker="s", label="ridge, alpha = 10")
ax.set_xlabel("number of training listings")
ax.set_ylabel("test R²")
ax.legend()
plt.show()

**Think.** What happens to the gap between the two models as the number of training listings grows? Why?

*Your answer:*

## Task 6. Back to your friend

Time to answer her question again. Her apartment, as a row with the seven features of Task 4: a whole home for 2 guests, one bedroom and one bed, 2.2 km from Dam Square. She has no reviews yet, so there is no rating: we fill in 4.9, the typical rating in Amsterdam.

**Run.**

In [ ]:
her7 = pd.DataFrame({"accommodates": [2], "bedrooms": [1], "beds": [1], "whole_home": [1],
                     "review_scores_rating": [4.9], "number_of_reviews": [0],
                     "dist_center_km": [float(np.sqrt(her["x_km"]**2 + her["y_km"]**2).iloc[0])]})

print("Lab 1, median of similar listings:   300")
print("k-NN (Task 3):                      ", round(grid_knn.predict(her)[0]))
print("linear regression, 7 features:      ", round(lr7.predict(her7)[0]))
print("typical error of the linear model:  ", round(np.abs(y_test - lr7.predict(X_test)).median()), "euros (median absolute residual)")

**Think.** What price would you tell her, and how sure are you? Three things to weigh: the three answers, the typical error, and the funnel of Task 4. And one more: we filled in a rating of 4.9 for a listing that has no rating. Why is that a problem for a model that uses the rating?

*Your answer:*

## Check your understanding

Questions 1 to 5 are the exam-style questions of Lectures 3 and 4. Questions 6 to 13 are in the format of earlier exams of this course (2017 to 2020): multiple choice with four options, pairs of true/false statements, questions about a figure, and an open question. Several are taken from those exams, sometimes adapted to Airbnb.

Answer each question for yourself first, then unfold the answer. We go through them together at the end of the lab.

**1.** Why not choose $k$ by its accuracy on the test set?

A. it is too slow &nbsp; B. the test accuracy becomes too optimistic &nbsp; C. scikit-learn forbids it

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** Once the test set chooses $k$, it is part of the training process, so its accuracy no longer estimates the accuracy on new data. In Task 2: 85.6% on the test set at $k = 34$, against an honest 84.9%.

</details>

**2.** In 5-fold cross-validation, each training example is used for validation

A. never &nbsp; B. exactly once &nbsp; C. five times

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** Each example sits in exactly one of the five folds, and each fold is the validation set once.

</details>

**3.** The data contain 50 schools with many students each. The model must work for new schools. Use

A. a random split &nbsp; B. `TimeSeriesSplit` &nbsp; C. `GroupKFold` by school

<details><summary><b>Answer</b> (click to unfold)</summary>

**C.** With a random split, students of the same school end up in training and validation, and the model is tested on schools it already knows. "Split the data the way the model will meet new data" (Lecture 3). On Airbnb: listings of the same host (see "If you finish early").

</details>

**4.** The test $R^2$ of a model is negative. The model

A. has a bug &nbsp; B. is worse than predicting the mean &nbsp; C. is perfect

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** $R^2 = 0$ means as good as predicting the mean; below 0 is worse. 1-NN on the data of Task 3 is an example: its test $R^2$ is about −0.05.

</details>

**5.** A $k$-NN classifier labels 90% of the test examples correctly. Is it a good model?

<details><summary><b>Answer</b> (click to unfold)</summary>

It depends on the baseline. If 90% of the examples belong to one class, a model that always predicts that class also gets 90%: then this $k$-NN has learned nothing. Always compare with the share of the largest class (Task 1).

</details>

**6.** Look at your plot "The effect of $k$" in Task 1. With $k = 1$, the classifier is likely

A. underfitting &nbsp; B. overfitting &nbsp; C. generalizing well &nbsp; D. none of the above

*(Exam of December 2018, question 1, with our plot.)*

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** Almost 100% on the training data, about 80% on the test data: the model follows single listings, noise included. On the exam, the figure is given; here you drew it yourself.

</details>

**7.** Sort these $k$-NN classifiers in increasing order of model complexity: (a) $k = 30$, (b) $k = 1$, (c) $k = 5$.

A. a → b → c &nbsp; B. a → c → b &nbsp; C. b → c → a &nbsp; D. c → b → a

*(As in the exam of December 2017, question 4, where the figure showed three decision boundaries.)*

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** A larger $k$ averages over more neighbors and gives a smoother, simpler boundary (Lecture 3, "The sweet spot": small $k$ = complex model). So $k = 30$ is the simplest and $k = 1$ the most complex. On the exam you recognize them by their boundaries: the more ragged, the smaller $k$.

</details>

**8.** Which of the following statements are true?

(a) An accuracy of 0.75 on the training set and 0.73 on the test set is a likely sign of underfitting. &nbsp; A. True &nbsp; B. False<br>
(b) An accuracy of 0.95 on the training set and 0.63 on the test set is a likely sign of overfitting. &nbsp; A. True &nbsp; B. False

*(Exam of December 2017, question 6.)*

<details><summary><b>Answer</b> (click to unfold)</summary>

(a) **True**: both scores are low and close together, so the model is too simple to fit even the training data. (In Task 1, 0.75 would hardly beat the baseline of 0.72.) (b) **True**: a large gap between training and test is the mark of overfitting, as for 1-NN in Task 1. On these exams, each true/false statement counted for one point.

</details>

**9.** Ridge regression is fitted for many values of $\alpha$. Which of the following statements are true?

(a) The strength of the regularization increases with $\alpha$. &nbsp; A. True &nbsp; B. False<br>
(b) Lower values of $\alpha$ yield a simpler model. &nbsp; A. True &nbsp; B. False

*(Exam of December 2018, question 2, about a figure of coefficient paths.)*

<details><summary><b>Answer</b> (click to unfold)</summary>

(a) **True**: $\alpha$ multiplies the penalty $\sum_j w_j^2$. (b) **False**: a *higher* $\alpha$ gives smaller coefficients and a simpler model; with $\alpha$ close to 0, ridge becomes ordinary least squares. With a very large $\alpha$, all coefficients are close to zero and the model underfits.

</details>

**10.** Which linear model for regression is given by the following equation?

$$\min_{w,\,b} \; \sum_{i=1}^{n} \big(y_i - \hat{y}_i\big)^2 + \alpha \sum_{j=1}^{p} |w_j|$$

A. Logistic regression with an L1 penalty &nbsp; B. Ordinary least squares &nbsp; C. Ridge &nbsp; D. Lasso

*(Retake of July 2018, question 1.)*

<details><summary><b>Answer</b> (click to unfold)</summary>

**D.** Squared gaps plus $\alpha$ times the sum of the *absolute* coefficients: lasso (Lecture 4). Ridge would have $\sum_j w_j^2$, ordinary least squares no penalty at all, and logistic regression is a classifier (Lecture 5).

</details>

**11.** A data scientist predicts the exact number of views that an Airbnb listing will get, from a large number of features. Interpretability is of utmost importance. Which model is likely to give the most interpretable result?

A. Logistic regression &nbsp; B. Ordinary least squares &nbsp; C. Ridge regression &nbsp; D. Lasso regression

*(Exam of December 2018, question 3, about the views of a product image.)*

<details><summary><b>Answer</b> (click to unfold)</summary>

**D.** Lasso sets most coefficients exactly to zero, so only a few features remain to explain, as in Task 5 (13 of 86). Ordinary least squares and ridge keep every feature; logistic regression predicts a class, not a number of views.

</details>

**12.** Which of the following is *not* a metric for regression?

A. Accuracy &nbsp; B. Coefficient of determination ($R^2$) &nbsp; C. Mean squared error &nbsp; D. Mean absolute error

*(Retake of July 2018, question 7.)*

<details><summary><b>Answer</b> (click to unfold)</summary>

**A.** Accuracy counts correct classes, so it is for classification. $R^2$ is the `score` of a regressor; the mean squared error is the average of the squared residuals, the mean absolute error the average of their absolute values. Task 6 used a close cousin: the median absolute residual.

</details>

**13. Open question: linear models for regression.** *(After the resit of July 2020, question 2.)* Answer in a few sentences, as on the exam:

- What does ordinary least squares minimize?
- Which term is it missing, and what is the drawback?
- How does ridge regression correct for this, and what does lasso do differently?
- How do you choose the strength of the penalty?

<details><summary><b>Answer</b> (click to unfold)</summary>

A model answer:

- **Ordinary least squares** chooses the coefficients and the intercept with the smallest sum of squared gaps between the actual and the predicted values: $\min \sum_i (y_i - \hat{y}_i)^2$. Every feature gets one coefficient.
- It has **no penalty term**, nothing that keeps the coefficients small. With many features and few examples it fits the noise in the training data: it overfits, and its coefficients can be large and unstable, especially for correlated features (Task 5: training $R^2$ 0.69, test 0.39).
- **Ridge** adds $\alpha \sum_j w_j^2$: all coefficients shrink towards zero, the model becomes simpler and generalizes better. **Lasso** adds $\alpha \sum_j |w_j|$ instead: many coefficients become exactly zero, so lasso also selects features, which makes the model easier to interpret. Both become ordinary least squares when $\alpha = 0$.
- $\alpha$ is a **hyperparameter**: choose it with cross-validation on the training set (`GridSearchCV` over, for example, `np.logspace(-2, 4, 25)`), and score the chosen model once on the test set.

</details>

## If you finish early

You can also do these at home, and compare with the solutions on Canvas.

**A. Listings of the same host.** One in four listings belongs to a host with more than one listing in Amsterdam, up to 24. Their listings are often alike, so a random split puts near-copies in the training and the validation folds. Lecture 3, "When a random split fails": keep each group in one fold with `GroupKFold`.

**Run.** The cross-validation $R^2$ of $k$-NN from Task 3, with folds made at random and with folds made by host.

In [ ]:
from sklearn.model_selection import KFold, GroupKFold

Xk = clean[["x_km", "y_km", "accommodates", "whole_home"]]
for k in [1, 13]:
    model = KNeighborsRegressor(n_neighbors=k)
    random_folds = cross_val_score(model, Xk, clean["price"], cv=KFold(5, shuffle=True, random_state=0))
    host_folds = cross_val_score(model, Xk, clean["price"], cv=GroupKFold(5), groups=clean["host_id"])
    print(f"k = {k:2d}: random folds R² {random_folds.mean():.3f},  folds by host R² {host_folds.mean():.3f}")

**Think.** Which estimate is more honest for a *new* host, such as your friend, and why is it lower?

*Your answer:*

**B. Your own model.** Try to beat the test $R^2$ of `lr7` (Task 4) with a linear model of your own: add or remove features from `clean`, for example `minimum_nights`, `availability_365` or `host_listings`. Keep the recipe: choose on the training set, score once on the test set.

In [ ]:
# my features: ...

## Where this is going

> $k$-NN predicts from similar examples; a linear model gives every feature one coefficient. Choose the hyperparameter, $k$ or $\alpha$, with cross-validation, and touch the test set only once.

Today you trained your first models, and you tested them honestly. You saw that an accuracy means little without a baseline, that $k$ trades overfitting against underfitting, that the units of the features decide what "near" means, that a linear model is easy to read but not easy to read *correctly*, and that a penalty helps when the features are many and the data few.

**Coming up.**

- Lecture 5 (Monday, November 9): linear models for classification.
- Lecture 6 (Tuesday, November 10): preprocessing and feature engineering: one scale for all features, text columns as numbers, the logarithm of the price.

**Reading.** Müller and Guido (2016), Chapter 2: "Classification and Regression", "Generalization, Overfitting, and Underfitting", "k-Nearest Neighbors" and "Linear Models", pp. 47–58; Chapter 5: "Cross-Validation" and "Grid Search".

---
*This lab practices Lectures 3 and 4 of Ş. İlker Birbil. Data: [Inside Airbnb](https://insideairbnb.com), Amsterdam, collected on September 16, 2026, licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/); downloaded directly from their website and cleaned as in Lab 1.*